# NB_Silver_Analyze_Load_ERP_Tables
Scope: promote ERP tables from **Bronze** to **Silver** Lakehouse.

**Flows:**
1. Reading csv configuration file and filtering for ERP tables
2. Quality check pre-cleaning
3. Cleaning function
4. Write data to Silver (Full Load)
5. Save Audit Log

##  Import & Parameters

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType, TimestampType
)
from delta.tables import DeltaTable
from datetime import datetime
import traceback

spark = SparkSession.builder.getOrCreate()

BRONZE_LAKEHOUSE = "LH_Bronze"
SILVER_LAKEHOUSE = "LH_Silver"
SOURCE_FOLDER    = "erp"
CONFIG_TABLE     = "pipeline_config"
AUDIT_TABLE      = "silver_audit_log"

# Columns name for the audit log file
COL_SILVER_TS  = "silver_processed_ts"   # timestamp writing into Silver
COL_SILVER_SRC = "silver_source_folder"  # (es. "erp")

# Relative path relativo csv configuration file in the current Lakehouse (Files/)
config_path = "Files/config_ingestion.csv"

## ABFSS path

In [ ]:
bronze_info = notebookutils.lakehouse.get(BRONZE_LAKEHOUSE)
bronze_path = bronze_info["properties"]["abfsPath"]


silver_info = notebookutils.lakehouse.get(SILVER_LAKEHOUSE)
silver_path = silver_info["properties"]["abfsPath"]

print(f"📂 Bronze path : {bronze_path}")
print(f"📂 Silver path : {silver_path}")

## Read config files and filter ERP tables

In [ ]:
df_config = (
    spark.read
    .option("header",      "true")
    .option("inferSchema", "true")
    .option("sep",         ";")
    .csv(config_path)
)

display(df_config)

rows = (
    df_config
    .filter(df_config.SourceFolder == SOURCE_FOLDER)
    .collect()
)

object_names = [row["DestinationTable"] for row in rows]

print(f"📋 Tabelle da processare: {len(object_names)} → {object_names}")

## Define functions (quality check & cleaning)

In [ ]:
# Cleaning Function
def applica_pulizia_superficiale(df: DataFrame, nome_tabella: str) -> DataFrame:

    # Identify only string column to apply trim
    colonne_stringa = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, StringType)
    ]

    # 1. TRIM: removing leading and trailing space from each string column
    for col_name in colonne_stringa:
        df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # 2. EMPTY → NULL: normalizi empty string as NULL 
    for col_name in colonne_stringa:
        df = df.withColumn(
            col_name,
            F.when(F.col(col_name) == "", None).otherwise(F.col(col_name))
        )

    # 3.    GHOST ROWS: eliminate rows where each column is NULL
    df = df.dropna(how="all")

    # 4. AUDIT COLUMN: add metadata to Silver tarceability
    df = (
        df
        .withColumn(COL_SILVER_TS,  F.current_timestamp())  
        .withColumn(COL_SILVER_SRC, F.lit(SOURCE_FOLDER))   
    )

    return df

# Quality report function
def calcola_quality_report(df: DataFrame, nome_tabella: str) -> dict:

    totale_righe    = df.count()

    # Duplicated rows
    righe_duplicate = totale_righe - df.dropDuplicates().count()

    # Ghost rows
    righe_full_null = totale_righe - df.dropna(how="all").count()

    # NULL % for each single column
    null_counts = df.select([
        F.round(
            (F.sum(F.col(c).isNull().cast("int")) / totale_righe) * 100, 2
        ).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()

    return {
        "tabella"         : nome_tabella,
        "totale_righe"    : totale_righe,
        "righe_duplicate" : righe_duplicate,
        "righe_full_null" : righe_full_null,
        "null_pct_per_col": null_counts
    }

## Apply Quality Check function

In [ ]:

print("=" * 65)
print("🔍 QUALITY CHECK PRE-PULIZIA")
print("=" * 65)

quality_reports = {}

for row in rows:
    nome_file    = row["ObjectName"]      
    nome_tabella = row["DestinationTable"]  

    try:
        # Reading via path ABFSS to avoid cross-lakehouse limit in the spark catalog
        df_bronze = (
            spark.read
            .format("delta")
            .load(f"{bronze_path}/Tables/{nome_tabella}")
        )

        report = calcola_quality_report(df_bronze, nome_tabella)
        quality_reports[nome_tabella] = report

        # print results with an attention threshold of > 10% NULL values for column
        print(f"\n📊 Tabella: {nome_tabella.upper()}")
        print(f"   ├── Righe totali    : {report['totale_righe']:,}")
        print(f"   ├── Righe duplicate : {report['righe_duplicate']:,} "
              f"({'⚠ ATTENZIONE' if report['righe_duplicate'] > 0 else '✅ OK'})")
        print(f"   ├── Righe full-null : {report['righe_full_null']:,} "
              f"({'⚠ ATTENZIONE' if report['righe_full_null'] > 0 else '✅ OK'})")
        print(f"   └── Null % per colonna:")
        for col_name, pct in report["null_pct_per_col"].items():
            flag = "⚠" if pct > 10 else "✅"
            print(f"         {flag} {col_name}: {pct}%")

    except Exception as e:
        print(f"\n❌ Errore lettura Bronze.{nome_tabella}: {e}")
        quality_reports[nome_tabella] = {"errore": str(e)}

print("\n" + "=" * 65)

## Write to Silver

In [ ]:
risultati = []

print("=" * 65)
print("🚀 INIZIO PROCESSING BRONZE → SILVER")
print("=" * 65)

for row in rows:
    nome_tabella = row["DestinationTable"]
    nome_silver  = nome_tabella.replace("bronze_", "silver_")
    inizio       = datetime.now()

    print(f"\n⏳ Processing: {nome_tabella} → {nome_silver}")

    try:
        # Read Bronze
        df_bronze    = spark.read.format("delta").load(f"{bronze_path}/Tables/{nome_tabella}")
        righe_bronze = df_bronze.count()

        # Apply Cleaning function
        df_silver    = applica_pulizia_superficiale(df_bronze, nome_tabella)
        righe_silver = df_silver.count()

        # Write to silver, overwrite mode
        (
            df_silver.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true") # the true allows to update the schema if it changes in the Bronze Lakehouse
            .save(f"{silver_path}/Tables/{nome_silver}")
        )

        fine     = datetime.now()
        durata_s = round((fine - inizio).total_seconds(), 2)

        print(f"   ✅ Completato in {durata_s}s")
        print(f"   ├── Righe Bronze  : {righe_bronze:,}")
        print(f"   ├── Righe Silver  : {righe_silver:,}")
        print(f"   └── Scartate      : {righe_bronze - righe_silver:,}")

        risultati.append({
            "tabella"       : nome_silver,
            "righe_bronze"  : righe_bronze,
            "righe_silver"  : righe_silver,
            "righe_scartate": righe_bronze - righe_silver,
            "durata_sec"    : durata_s,
            "stato"         : "SUCCESS",
            "errore"        : "",      # use empty string, otherwise None would break the schema StringType
            "processed_ts"  : fine
        })

    except Exception as e:
        fine = datetime.now()
        msg  = traceback.format_exc()
        print(f"   ❌ ERRORE su {nome_tabella}: {e}")

        risultati.append({
            "tabella"       : nome_silver,
            "righe_bronze"  : 0,
            "righe_silver"  : 0,
            "righe_scartate": 0,
            "durata_sec"    : round((fine - inizio).total_seconds(), 2),
            "stato"         : "FAILED",
            "errore"        : msg,
            "processed_ts"  : fine
        })
        continue  # continue with the next table without break the loop

print("\n" + "=" * 65)
print("🏁 PROCESSING COMPLETATO")
print("=" * 65)

## Summary result & audit logs

In [ ]:
# Explicit schema for the audit log
schema_audit = StructType([
    StructField("tabella",        StringType(),    True),
    StructField("righe_bronze",   IntegerType(),   True),
    StructField("righe_silver",   IntegerType(),   True),
    StructField("righe_scartate", IntegerType(),   True),
    StructField("durata_sec",     FloatType(),     True),
    StructField("stato",          StringType(),    True),
    StructField("errore",         StringType(),    True),
    StructField("processed_ts",   TimestampType(), True),
])

successi = [r for r in risultati if r["stato"] == "SUCCESS"]
falliti  = [r for r in risultati if r["stato"] == "FAILED"]

print(f"\n📊 RIEPILOGO ESECUZIONE")
print(f"   ✅ Tabelle OK     : {len(successi)}/{len(risultati)}")
print(f"   ❌ Tabelle Fallite: {len(falliti)}/{len(risultati)}")

if falliti:
    print(f"\n⚠  TABELLE CON ERRORI:")
    for r in falliti:
        print(f"   → {r['tabella']}: {r['errore'][:120]}...")


# Saving result in audit lof tables with appending mode
df_audit = spark.createDataFrame(risultati, schema=schema_audit)

(
    df_audit.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(f"{silver_path}/Tables/{AUDIT_TABLE}")
)

print(f"\n📝 Audit log salvato in: Silver → {AUDIT_TABLE}")

# Manage failing raising an exception
if falliti:
    raise Exception(
        f"❌ {len(falliti)} tabelle non processate. "
        f"Consulta '{AUDIT_TABLE}' per i dettagli."
    )